In [1]:
# Import packages
import os
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions
from mediapipe.tasks.python import BaseOptions
import pandas as pd
from tqdm import tqdm
import urllib.request

In [2]:
# Configure directories
UCF_VIDEOS = "UCF-101"
UCF_SPLITS = "UCF101TrainTestSplits/ucfTrainTestlist"
FRAMES_DIR = "outputs/frames"
KEYPOINTS_DIR = "outputs/keypoints"

# Convert UCF101 classes to our labels
EXERCISE_CLASSES = {
    "PushUps": "pushups", 
    "BodyWeightSquats": "squats",
    "Lunges": "lunges",
    "PullUps": "pullups",
    "JumpingJack": "jumping_jacks"
}

# Configure desired frame rates and image size
TARGET_FPS = 10
IMG_SIZE = (640, 480)

In [3]:
# Filter 5 classes from train/test 01
# We are just using one train/test split for this project

def load_videos():
    rows = []
    for split, filename in [("train", "trainlist01.txt"), ("test", "testlist01.txt")]:
        filepath = os.path.join(UCF_SPLITS, filename)
        with open(filepath) as f:
            for line in f: # e.g. BodyWeightSquats/v_BodyWeightSquats_g08_c01.avi
                rel_path = line.strip().split()[0]
                class_name = rel_path.split("/")[0]
                if class_name not in EXERCISE_CLASSES:
                    continue
                rows.append({
                    "video_path": os.path.join(UCF_VIDEOS, rel_path),
                    "label": EXERCISE_CLASSES[class_name],
                    "split": split,
                    "video_name": os.path.splitext(os.path.basename(rel_path))[0]
                })
    df = pd.DataFrame(rows)
    print(f"Found {len(df)} exercise videos\n{df['label'].value_counts()}\n")
    return df

In [4]:
# Extract frames from the videos
def extract_frames(video_df):
    rows = []
    for _, row in tqdm(video_df.iterrows(), total=len(video_df), desc="Extracting keyframes"):
        save_dir = os.path.join(FRAMES_DIR, row["split"], row["label"], row["video_name"])
        os.makedirs(save_dir, exist_ok=True)

        # Use OpenCV to open the .avi file
        # Checks the video's native FPS and determines how many frames to get based on that
        cap = cv2.VideoCapture(row["video_path"])
        fps = cap.get(cv2.CAP_PROP_FPS) or TARGET_FPS
        frame_interval = max(1, int(round(fps / TARGET_FPS)))

        frame_idx = 0 # Tracks every frame seens
        saved_idx = 0 # Tracks only the frames to save

        while True:
            ret, frame = cap.read()
            if not ret:
                break # Ret = False once the video has ended, so break
            if frame_idx % frame_interval == 0: # Resize and save frames in wanted interval
                frame_path = os.path.join(save_dir, f"frame_{saved_idx:04d}.jpg")
                cv2.imwrite(frame_path, cv2.resize(frame, IMG_SIZE))
                rows.append({**row, "frame_path": frame_path, "frame_index": saved_idx})
                saved_idx += 1
            frame_idx += 1
        cap.release() # Closes the video
    df = pd.DataFrame(rows)
    print(f"Extracted {len(df)} frames total\n")
    return df

In [5]:
# Name keypoints according to MediaPipe Pose standards
KEYPOINT_NAMES = [
    "nose", "left_eye_inner", "left_eye", "left_eye_outer",
    "right_eye_inner", "right_eye", "right_eye_outer",
    "left_ear", "right_ear", "mouth_left", "mouth_right",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_pinky", "right_pinky",
    "left_index", "right_index", "left_thumb", "right_thumb",
    "left_hip", "right_hip", "left_knee", "right_knee",
    "left_ankle", "right_ankle", "left_heel", "right_heel",
    "left_foot_index", "right_foot_index",
]

# Download the MediaPipe Pose model
MODEL_PATH = "pose_landmarker.task"
if not os.path.exists(MODEL_PATH):
    print("Downloading pose model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
        MODEL_PATH
    )
    print("Downloaded!")

# Extract keypoints 
def extract_keypoints(frame_df):
    os.makedirs(KEYPOINTS_DIR, exist_ok=True)
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
        num_poses=1,
        min_pose_detection_confidence=0.5,
    )
    rows = []
    with PoseLandmarker.create_from_options(options) as landmarker:
        # Loop through each frame in the dataframe
        for _, row in tqdm(frame_df.iterrows(), total=len(frame_df), desc="Pose estimation"):
            frame = cv2.imread(row["frame_path"])
            if frame is None:
                continue
            
            # Run MediaPipe on the frame after converting to RGB
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            results = landmarker.detect(mp_image)
            if not results.pose_landmarks:
                continue

            landmarks = results.pose_landmarks[0]
            keypoints = {}
            for i, name in enumerate(KEYPOINT_NAMES):
                kp = landmarks[i]
                # x and y-> position on screen
                keypoints[f"{name}_x"]   = round(kp.x, 6)
                keypoints[f"{name}_y"]   = round(kp.y, 6)

            rows.append({
                "frame_path": row["frame_path"],
                "video_name": row["video_name"],
                "label": row["label"],
                "split": row["split"],
                "frame_index": row["frame_index"],
                **keypoints # Append entire keyframes dictionary at once
            })

    df = pd.DataFrame(rows)
    # Output two CSVs that the model will train on (train and test keypoints)
    for split, group in df.groupby("split"):
        out_path = os.path.join(KEYPOINTS_DIR, f"{split}_keypoints.csv")
        group.to_csv(out_path, index=False)
        print(f"Saved {len(group)} rows → {out_path}")
        print(group["label"].value_counts(), "\n")
    return df

In [6]:
video_df = load_videos()
frame_df = extract_frames(video_df)
extract_keypoints(frame_df)

Found 564 exercise videos
label
lunges           127
jumping_jacks    123
squats           112
pushups          102
pullups          100
Name: count, dtype: int64



Extracting keyframes: 100%|██████████| 564/564 [01:49<00:00,  5.13it/s]


Extracted 34043 frames total



Pose estimation: 100%|██████████| 34043/34043 [24:08<00:00, 23.50it/s] 


Saved 7652 rows → outputs/keypoints\test_keypoints.csv
label
lunges           2952
squats           1812
pushups           980
jumping_jacks     971
pullups           937
Name: count, dtype: int64 

Saved 21289 rows → outputs/keypoints\train_keypoints.csv
label
lunges           8564
squats           5312
pullups          2819
pushups          2311
jumping_jacks    2283
Name: count, dtype: int64 



,frame_path,video_name,label,split,frame_index,nose_x,nose_y,left_eye_inner_x,left_eye_inner_y,left_eye_x,...,right_ankle_x,right_ankle_y,left_heel_x,left_heel_y,right_heel_x,right_heel_y,left_foot_index_x,left_foot_index_y,right_foot_index_x,right_foot_index_y
0,outputs/frames\train\squats\v_BodyWeightSquats...,v_BodyWeightSquats_g08_c01,squats,train,0,0.498673,0.326496,0.498823,0.310232,0.500799,...,0.418064,0.926531,0.502136,0.951608,0.416589,0.943314,0.521340,0.987345,0.437218,0.987540
1,outputs/frames\train\squats\v_BodyWeightSquats...,v_BodyWeightSquats_g08_c01,squats,train,1,0.500554,0.326976,0.500588,0.310929,0.502863,...,0.416381,0.922342,0.515106,0.950867,0.417111,0.942610,0.539033,0.987205,0.431652,0.986853
2,outputs/frames\train\squats\v_BodyWeightSquats...,v_BodyWeightSquats_g08_c01,squats,train,2,0.499823,0.333185,0.499327,0.315805,0.501402,...,0.418781,0.929287,0.501734,0.939595,0.416618,0.949759,0.534167,0.994335,0.434127,0.993849
3,outputs/frames\train\squats\v_BodyWeightSquats...,v_BodyWeightSquats_g08_c01,squats,train,3,0.498956,0.325757,0.499174,0.310002,0.501396,...,0.416944,0.929216,0.502690,0.939821,0.408781,0.947273,0.539880,0.989181,0.434386,0.992563
4,outputs/frames\train\squats\v_BodyWeightSquats...,v_BodyWeightSquats_g08_c01,squats,train,4,0.502800,0.319947,0.501783,0.304821,0.503626,...,0.414609,0.930650,0.514345,0.958497,0.416563,0.932616,0.542549,0.990175,0.433984,0.993992
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28936,outputs/frames\test\pushups\v_PushUps_g07_c04\...,v_PushUps_g07_c04,pushups,test,33,0.491798,0.741182,0.504473,0.722548,0.511692,...,0.463961,0.596942,0.560517,0.595279,0.465687,0.575658,0.560055,0.663198,0.476345,0.633492
28937,outputs/frames\test\pushups\v_PushUps_g07_c04\...,v_PushUps_g07_c04,pushups,test,34,0.459866,0.624580,0.474851,0.611060,0.480182,...,0.450235,0.766787,0.518398,0.703976,0.450736,0.755018,0.536200,0.724278,0.461990,0.794594
28938,outputs/frames\test\pushups\v_PushUps_g07_c04\...,v_PushUps_g07_c04,pushups,test,36,0.485839,0.422360,0.497983,0.404252,0.503457,...,0.421034,0.497498,0.525097,0.510711,0.430590,0.484257,0.556674,0.577655,0.427744,0.554614
28939,outputs/frames\test\pushups\v_PushUps_g07_c04\...,v_PushUps_g07_c04,pushups,test,37,0.474144,0.419561,0.489614,0.402014,0.495636,...,0.405623,0.473469,0.535121,0.495098,0.414505,0.468439,0.567136,0.551393,0.405220,0.512244


In [7]:
# Delete unnecessary keypoints so file size can be saved to GitHub
useful_keypoints = [
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle", "nose"
]

train_df = pd.read_csv("outputs/keypoints/train_keypoints.csv")
keep_cols = ["frame_path", "video_name", "label", "split", "frame_index"]
for kp in useful_keypoints:
    keep_cols += [f"{kp}_x", f"{kp}_y"]

train_df = train_df[keep_cols]
train_df.to_csv("outputs/keypoints/train_keypoints.csv", index=False)

test_df = pd.read_csv("outputs/keypoints/test_keypoints.csv")
keep_cols = ["frame_path", "video_name", "label", "split", "frame_index"]
for kp in useful_keypoints:
    keep_cols += [f"{kp}_x", f"{kp}_y"]

test_df = test_df[keep_cols]
test_df.to_csv("outputs/keypoints/test_keypoints.csv", index=False)

## Feature Computation and Labeling

In [9]:
def compute_angle(a, b, c):
    """Angle at point b, formed by a-b-c"""
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))

def add_angles(df):
    def angle(r, a, b, c):
        """Helper to pull x/y from row and compute angle at b"""
        return compute_angle(
            (r[f"{a}_x"], r[f"{a}_y"]),
            (r[f"{b}_x"], r[f"{b}_y"]),
            (r[f"{c}_x"], r[f"{c}_y"])
        )

    # --- KNEE ANGLES (squats, lunges) ---
    for side in ["left", "right"]:
        df[f"{side}_knee_angle"] = df.apply(
            lambda r, s=side: angle(r, f"{s}_hip", f"{s}_knee", f"{s}_ankle"), axis=1)

    # --- ELBOW ANGLES (push-ups, pull-ups) ---
    for side in ["left", "right"]:
        df[f"{side}_elbow_angle"] = df.apply(
            lambda r, s=side: angle(r, f"{s}_shoulder", f"{s}_elbow", f"{s}_wrist"), axis=1)

    # --- HIP ANGLES (squats, lunges, pull-ups) ---
    for side in ["left", "right"]:
        df[f"{side}_hip_angle"] = df.apply(
            lambda r, s=side: angle(r, f"{s}_shoulder", f"{s}_hip", f"{s}_knee"), axis=1)

    # --- BODY LINE / HIP SAG (push-ups) ---
    for side in ["left", "right"]:
        df[f"{side}_body_line"] = df.apply(
            lambda r, s=side: angle(r, f"{s}_shoulder", f"{s}_hip", f"{s}_ankle"), axis=1)

    # --- TORSO UPRIGHTNESS (lunges, squats) ---
    # Angle between vertical and the shoulder→hip vector
    df["torso_angle"] = df.apply(lambda r: compute_angle(
        (r["left_shoulder_x"], r["left_shoulder_y"] - 0.1),  # point above shoulder
        (r["left_shoulder_x"], r["left_shoulder_y"]),
        (r["left_hip_x"],      r["left_hip_y"])
    ), axis=1)

    # --- KNEE VALGUS (squats, lunges) ---
    # How far knee is inside/outside the ankle — not an angle, just a diff
    for side in ["left", "right"]:
        df[f"{side}_knee_valgus"] = df[f"{side}_knee_x"] - df[f"{side}_ankle_x"]

    # --- HEAD DROP (push-ups) ---
    df["head_drop"] = df["nose_y"] - df["left_shoulder_y"]

    # --- SHOULDER SHRUG (pull-ups) ---
    for side in ["left", "right"]:
        df[f"{side}_shoulder_shrug"] = df[f"{side}_ear_y"] - df[f"{side}_shoulder_y"]

    # --- SYMMETRY (jumping jacks) ---
    df["arm_symmetry"] = abs(df["left_elbow_angle"] - df["right_elbow_angle"])
    df["leg_symmetry"] = abs(df["left_knee_angle"]  - df["right_knee_angle"])
    df["hip_symmetry"] = abs(df["left_hip_angle"]   - df["right_hip_angle"])

    return df